# Lab C — Medallion Data and AI Foundation

## Enterprise Contract Intelligence — Bronze → Silver → Gold

```text
Enterprise Sources
     │
     ├── Supplier MDM
     ├── Contract Master
     ├── Contract Documents
     ├── Ariba Metadata
     ├── ERP Payments
     └── SharePoint Metadata
     │
     ▼
┌──────────────────────────────┐
│ BRONZE — Raw Enterprise Data │
│                              │
│ bronze_suppliers             │
│ bronze_contracts             │
│ bronze_documents             │
│ bronze_ariba                 │
│ bronze_erp_payments          │
│ bronze_sharepoint            │
└──────────────┬───────────────┘
               │
               ▼
        AI Extraction Layer
        ai_query + fallback
               │
               ▼
┌──────────────────────────────┐
│ SILVER — Contract Intelligence│
│                               │
│ silver_extractions            │
│ silver_obligations            │
│ silver_milestones             │
│ silver_payments               │
│ silver_risks                  │
│ silver_contract_clauses       │
└──────────────┬────────────────┘
               │
               ▼
      Enterprise Consolidation
               │
               ▼
┌──────────────────────────────┐
│ GOLD — Business Contract View│
│                              │
│ gold_enterprise_contracts    │
│                              │
│ supplier                     │
│ value / expiry               │
│ next milestone               │
│ next payment                 │
│ obligations                  │
│ SLA / penalty                │
│ renewal exposure             │
│ risk severity                │
└──────────────┬───────────────┘
               │
               ▼
 gold_enterprise_contract_view
 Materialized View if supported
        │
        ▼
Agent-ready foundation for Lab D
```

## Objective

Build the complete contract-intelligence data and AI foundation using the medallion architecture:

**Sources → Bronze → AI extraction / normalization → Silver → Gold → Materialized View**

## What this lab proves

- Raw enterprise contract data can be organized into an explicit Bronze layer.
- LLM-based extraction can convert contract text into governed structured intelligence.
- Extracted obligations, milestones, payments, risks, and clauses can be normalized into Silver.
- Silver data can be consolidated into a Gold enterprise contract operating view.
- Gold can expose a materialized view for downstream analytics and agents.
- Silver clause data remains available for retrieval and evidence grounding in Lab D.

## Free Edition approach

- Uses serverless-compatible Spark DataFrames and SQL.
- Attempts Databricks `ai_query`.
- Falls back to deterministic extraction when model access is unavailable.
- Avoids RDD, cache, persist, and cluster-specific APIs.
- Attempts a materialized view and falls back to a standard SQL view when unsupported.


## Section 1 — Configure Medallion Objects

In [0]:
# Discover the current catalog/schema and define explicit Bronze, Silver, and Gold table names.
row = spark.sql(
    "SELECT current_catalog() AS catalog_name, current_schema() AS schema_name"
).first()

CATALOG = row["catalog_name"]
SCHEMA = row["schema_name"]

# Bronze tables
BRONZE_SUPPLIERS = f"{CATALOG}.{SCHEMA}.bronze_suppliers"
BRONZE_CONTRACTS = f"{CATALOG}.{SCHEMA}.bronze_contracts"
BRONZE_DOCUMENTS = f"{CATALOG}.{SCHEMA}.bronze_documents"
BRONZE_ARIBA = f"{CATALOG}.{SCHEMA}.bronze_ariba"
BRONZE_ERP_PAYMENTS = f"{CATALOG}.{SCHEMA}.bronze_erp_payments"
BRONZE_SHAREPOINT = f"{CATALOG}.{SCHEMA}.bronze_sharepoint"

# Silver tables
SILVER_EXTRACTIONS = f"{CATALOG}.{SCHEMA}.silver_extractions"
SILVER_OBLIGATIONS = f"{CATALOG}.{SCHEMA}.silver_obligations"
SILVER_MILESTONES = f"{CATALOG}.{SCHEMA}.silver_milestones"
SILVER_PAYMENTS = f"{CATALOG}.{SCHEMA}.silver_payments"
SILVER_RISKS = f"{CATALOG}.{SCHEMA}.silver_risks"
SILVER_CONTRACT_CLAUSES = f"{CATALOG}.{SCHEMA}.silver_contract_clauses"

# Gold objects
GOLD_ENTERPRISE_CONTRACTS = f"{CATALOG}.{SCHEMA}.gold_enterprise_contracts"
GOLD_ENTERPRISE_CONTRACT_VIEW = f"{CATALOG}.{SCHEMA}.gold_enterprise_contract_view"

# AI configuration
LLM_ENDPOINT = "system.ai.meta-llama-3-3-70b-instruct"
RUN_LLM_EXTRACTION = True

print("Catalog:", CATALOG)
print("Schema:", SCHEMA)
print("LLM endpoint:", LLM_ENDPOINT)


## Section 2 — Create Bronze Layer

In [0]:
# Create the Bronze supplier master table from synthetic enterprise source data.
import pandas as pd
import numpy as np
import random
from datetime import date, timedelta

random.seed(42)
np.random.seed(42)

suppliers = [
    ("SUP001", "Apex Industrial Systems", "Industrial Equipment", "India", "Strategic"),
    ("SUP002", "BluePeak Technology Services", "IT Services", "India", "Strategic"),
    ("SUP003", "Crestline Logistics", "Logistics", "Singapore", "Preferred"),
    ("SUP004", "Delta Process Automation", "Automation", "Germany", "Strategic"),
    ("SUP005", "Evergreen Facilities", "Facilities", "India", "Preferred"),
    ("SUP006", "Fusion Cloud Solutions", "Cloud Services", "USA", "Strategic"),
]

supplier_df = spark.createDataFrame(
    suppliers,
    [
        "supplier_id",
        "supplier_name",
        "category",
        "country",
        "supplier_tier"
    ]
)

supplier_df.write.mode("overwrite").format("delta").saveAsTable(
    BRONZE_SUPPLIERS
)

display(supplier_df)


In [0]:
# Generate Bronze contract, Ariba, ERP payment, SharePoint, and synthetic ground-truth records.
base_date = date(2026, 9, 1)

contract_rows = []
ariba_rows = []
erp_rows = []
sharepoint_rows = []
truth_rows = []

for i in range(1, 13):
    supplier_id, supplier_name, _, _, _ = suppliers[(i - 1) % len(suppliers)]

    contract_id = f"CTR-2026-{i:03d}"
    ariba_id = f"ARIBA-{8000 + i}"

    start_date = base_date - timedelta(days=180 + i * 7)
    end_date = base_date + timedelta(days=90 + i * 25)

    contract_value = float(4_000_000 + i * 1_350_000)
    currency = ["INR", "USD", "EUR"][i % 3]
    owner = ["Procurement", "Finance", "IT", "Operations"][i % 4]

    milestone_1 = f"Complete design and readiness review for work package {i}"
    milestone_2 = f"Complete production handover for work package {i}"
    milestone_1_date = base_date + timedelta(days=10 + i * 3)
    milestone_2_date = base_date + timedelta(days=40 + i * 4)

    payment_1 = round(contract_value * 0.35, 2)
    payment_2 = round(contract_value * 0.45, 2)
    payment_3 = round(contract_value * 0.20, 2)

    payment_1_date = milestone_1_date + timedelta(days=15)
    payment_2_date = milestone_2_date + timedelta(days=20)
    payment_3_date = end_date - timedelta(days=15)

    severity = ["LOW", "MEDIUM", "HIGH"][i % 3]
    risk_type = [
        "Delivery Delay",
        "SLA Exposure",
        "Renewal Risk",
        "Payment Dependency"
    ][i % 4]

    sla_target = 97 + (i % 3)
    delay_penalty_pct = round(0.25 + (i % 4) * 0.15, 2)
    renewal_notice_days = [30, 60, 90][i % 3]

    commitment = (
        f"Deliver contracted services and milestone outputs "
        f"for work package {i}."
    )

    recommended_action = {
        "LOW": "Track through normal contract governance.",
        "MEDIUM": "Request supplier status update and confirm owner action.",
        "HIGH": "Escalate to procurement owner and initiate mitigation review."
    }[severity]

    contract_rows.append(
        (
            contract_id,
            supplier_id,
            ariba_id,
            ["Ariba", "SharePoint", "Contract Repository"][i % 3],
            start_date.isoformat(),
            end_date.isoformat(),
            contract_value,
            currency,
            owner,
            "ACTIVE"
        )
    )

    ariba_rows.append(
        (
            ariba_id,
            contract_id,
            supplier_id,
            f"PO-{50000 + i}",
            owner,
            "Approved"
        )
    )

    for payment_no, milestone, amount, due_date in [
        (1, milestone_1, payment_1, payment_1_date),
        (2, milestone_2, payment_2, payment_2_date),
        (3, "Final acceptance", payment_3, payment_3_date),
    ]:
        erp_rows.append(
            (
                contract_id,
                payment_no,
                milestone,
                float(amount),
                due_date.isoformat(),
                "OPEN"
            )
        )

    sharepoint_rows.append(
        (
            contract_id,
            f"/Contracts/2026/{contract_id}.pdf",
            f"{contract_id}.pdf",
            (base_date - timedelta(days=i)).isoformat(),
            "Legal-Procurement"
        )
    )

    truth_rows.append(
        {
            "contract_id": contract_id,
            "supplier_name": supplier_name,
            "commitment": commitment,
            "milestone_1": milestone_1,
            "milestone_1_date": milestone_1_date.isoformat(),
            "milestone_2": milestone_2,
            "milestone_2_date": milestone_2_date.isoformat(),
            "payment_1": float(payment_1),
            "payment_1_date": payment_1_date.isoformat(),
            "payment_2": float(payment_2),
            "payment_2_date": payment_2_date.isoformat(),
            "payment_3": float(payment_3),
            "payment_3_date": payment_3_date.isoformat(),
            "currency": currency,
            "sla_target": int(sla_target),
            "delay_penalty_pct": float(delay_penalty_pct),
            "renewal_notice_days": int(renewal_notice_days),
            "risk_type": risk_type,
            "risk_severity": severity,
            "recommended_action": recommended_action
        }
    )

spark.createDataFrame(
    contract_rows,
    [
        "contract_id",
        "supplier_id",
        "ariba_contract_id",
        "source_system",
        "start_date",
        "end_date",
        "contract_value",
        "currency",
        "contract_owner",
        "status"
    ]
).write.mode("overwrite").format("delta").saveAsTable(
    BRONZE_CONTRACTS
)

spark.createDataFrame(
    ariba_rows,
    [
        "ariba_contract_id",
        "contract_id",
        "supplier_id",
        "purchase_order",
        "business_owner",
        "approval_status"
    ]
).write.mode("overwrite").format("delta").saveAsTable(
    BRONZE_ARIBA
)

spark.createDataFrame(
    erp_rows,
    [
        "contract_id",
        "payment_no",
        "linked_milestone",
        "amount",
        "due_date",
        "payment_status"
    ]
).write.mode("overwrite").format("delta").saveAsTable(
    BRONZE_ERP_PAYMENTS
)

spark.createDataFrame(
    sharepoint_rows,
    [
        "contract_id",
        "document_path",
        "file_name",
        "last_modified",
        "security_group"
    ]
).write.mode("overwrite").format("delta").saveAsTable(
    BRONZE_SHAREPOINT
)

truth_pdf = pd.DataFrame(truth_rows)

display(spark.table(BRONZE_CONTRACTS))


In [0]:
# Generate Bronze contract documents containing commitments, milestones, payments, SLA terms, penalties, renewals, and risks.
document_rows = []

for truth in truth_rows:
    contract_meta = next(
        row for row in contract_rows
        if row[0] == truth["contract_id"]
    )

    contract_text = f"""
SECTION 1 - CONTRACT OVERVIEW
Contract ID: {truth['contract_id']}
Supplier: {truth['supplier_name']}
Contract value: {contract_meta[6]:,.2f} {contract_meta[7]}
Effective date: {contract_meta[4]}
Expiry date: {contract_meta[5]}

SECTION 2 - SUPPLIER COMMITMENTS
The Supplier commits to: {truth['commitment']}
The supplier must maintain required staffing, delivery capacity, governance reporting, and evidence of completion.

SECTION 3 - MILESTONES
Milestone 1: {truth['milestone_1']}. Target date: {truth['milestone_1_date']}.
Milestone 2: {truth['milestone_2']}. Target date: {truth['milestone_2_date']}.
Final acceptance is required before contract closure.

SECTION 4 - PAYMENT SCHEDULE
Payment 1: {truth['payment_1']:,.2f} {truth['currency']} due on {truth['payment_1_date']} after milestone acceptance.
Payment 2: {truth['payment_2']:,.2f} {truth['currency']} due on {truth['payment_2_date']} after production handover.
Payment 3: {truth['payment_3']:,.2f} {truth['currency']} due on {truth['payment_3_date']} after final acceptance.
Payments may be placed on hold when linked milestone evidence is incomplete.

SECTION 5 - SERVICE LEVEL AND PENALTIES
The Supplier must maintain an SLA attainment of at least {truth['sla_target']} percent.
Delay beyond the agreed milestone date may attract a penalty of {truth['delay_penalty_pct']} percent of the affected milestone value per week.

SECTION 6 - RENEWAL AND TERMINATION
The contract requires renewal review at least {truth['renewal_notice_days']} days before expiry.
Material breach, repeated SLA failure, or non-delivery may trigger termination review.

SECTION 7 - IDENTIFIED RISK
Risk type: {truth['risk_type']}.
Risk severity: {truth['risk_severity']}.
Recommended action: {truth['recommended_action']}
""".strip()

    document_rows.append(
        (
            truth["contract_id"],
            f"{truth['contract_id']}.pdf",
            contract_text
        )
    )

document_df = spark.createDataFrame(
    document_rows,
    [
        "contract_id",
        "document_name",
        "contract_text"
    ]
)

document_df.write.mode("overwrite").format("delta").saveAsTable(
    BRONZE_DOCUMENTS
)

display(document_df.limit(3))


## Section 3 — Validate Bronze Layer

In [0]:
# Show Bronze layer row counts to confirm the raw enterprise foundation is complete.
bronze_counts = []

for logical_name, table_name in [
    ("bronze_suppliers", BRONZE_SUPPLIERS),
    ("bronze_contracts", BRONZE_CONTRACTS),
    ("bronze_documents", BRONZE_DOCUMENTS),
    ("bronze_ariba", BRONZE_ARIBA),
    ("bronze_erp_payments", BRONZE_ERP_PAYMENTS),
    ("bronze_sharepoint", BRONZE_SHAREPOINT),
]:
    bronze_counts.append(
        (
            logical_name,
            table_name,
            spark.table(table_name).count()
        )
    )

display(
    spark.createDataFrame(
        bronze_counts,
        ["layer_object", "table_name", "row_count"]
    )
)


## Section 4 — AI Extraction from Bronze Documents

The AI extraction step converts contract text into structured contract intelligence.

Target outputs:
- supplier commitment
- milestones
- payments
- SLA target
- delay penalty
- renewal notice
- risk type
- risk severity
- recommended action

The notebook attempts Databricks `ai_query` first.

If unavailable, it uses the synthetic ground-truth fallback so the Silver and Gold layers remain runnable.


In [0]:
# Define the strict JSON extraction prompt and deterministic fallback used for Silver-layer creation.
import json
import re

EXTRACTION_PROMPT = """
You are an enterprise contract intelligence extraction engine.

Read the contract below and return ONLY valid JSON.
Do not include markdown fences or commentary.

Return exactly:
{
  "supplier_name": "string",
  "commitment": "string",
  "milestones": [
    {
      "milestone": "string",
      "target_date": "YYYY-MM-DD"
    }
  ],
  "payments": [
    {
      "payment_no": 1,
      "linked_milestone": "string",
      "amount": 0.0,
      "currency": "string",
      "due_date": "YYYY-MM-DD"
    }
  ],
  "sla_target_pct": 0.0,
  "delay_penalty_pct_per_week": 0.0,
  "renewal_notice_days": 0,
  "risk_type": "string",
  "risk_severity": "LOW|MEDIUM|HIGH",
  "recommended_action": "string"
}

CONTRACT:
"""

def fallback_extract(contract_id):
    truth = truth_pdf.loc[
        truth_pdf["contract_id"] == contract_id
    ].iloc[0].to_dict()

    return {
        "supplier_name": truth["supplier_name"],
        "commitment": truth["commitment"],
        "milestones": [
            {
                "milestone": truth["milestone_1"],
                "target_date": truth["milestone_1_date"]
            },
            {
                "milestone": truth["milestone_2"],
                "target_date": truth["milestone_2_date"]
            }
        ],
        "payments": [
            {
                "payment_no": 1,
                "linked_milestone": truth["milestone_1"],
                "amount": truth["payment_1"],
                "currency": truth["currency"],
                "due_date": truth["payment_1_date"]
            },
            {
                "payment_no": 2,
                "linked_milestone": truth["milestone_2"],
                "amount": truth["payment_2"],
                "currency": truth["currency"],
                "due_date": truth["payment_2_date"]
            },
            {
                "payment_no": 3,
                "linked_milestone": "Final acceptance",
                "amount": truth["payment_3"],
                "currency": truth["currency"],
                "due_date": truth["payment_3_date"]
            }
        ],
        "sla_target_pct": truth["sla_target"],
        "delay_penalty_pct_per_week": truth["delay_penalty_pct"],
        "renewal_notice_days": truth["renewal_notice_days"],
        "risk_type": truth["risk_type"],
        "risk_severity": truth["risk_severity"],
        "recommended_action": truth["recommended_action"]
    }


In [0]:
# Attempt LLM extraction with ai_query and fall back automatically if the model endpoint is unavailable.
def clean_json_text(value):
    text = (value or "").strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.I)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

extractions = []
EXTRACTION_MODE = "deterministic_fallback"

if RUN_LLM_EXTRACTION:
    try:
        for row in spark.table(BRONZE_DOCUMENTS).collect():
            prompt = (
                EXTRACTION_PROMPT
                + "\n"
                + row["contract_text"]
            )

            prompt_sql = prompt.replace("'", "''")
            endpoint_sql = LLM_ENDPOINT.replace("'", "''")

            response = spark.sql(
                f"SELECT ai_query('{endpoint_sql}', '{prompt_sql}') AS response"
            ).first()["response"]

            parsed = json.loads(
                clean_json_text(response)
            )

            parsed["contract_id"] = row["contract_id"]
            parsed["extraction_method"] = "ai_query"

            extractions.append(parsed)

        EXTRACTION_MODE = "ai_query"

    except Exception as e:
        print("ai_query unavailable; using deterministic fallback.")
        print(type(e).__name__, str(e)[:500])
        extractions = []

if not extractions:
    for contract_id in truth_pdf["contract_id"].tolist():
        parsed = fallback_extract(contract_id)
        parsed["contract_id"] = contract_id
        parsed["extraction_method"] = "deterministic_fallback"
        extractions.append(parsed)

print("Extraction mode:", EXTRACTION_MODE)
print("Contracts extracted:", len(extractions))


## Section 5 — Build Silver Layer

In [0]:
# Persist the raw AI extraction JSON as the first Silver-layer contract intelligence table.
silver_extraction_df = spark.createDataFrame(
    [
        (
            item["contract_id"],
            item["extraction_method"],
            json.dumps(item)
        )
        for item in extractions
    ],
    [
        "contract_id",
        "extraction_method",
        "extraction_json"
    ]
)

silver_extraction_df.write.mode("overwrite").format("delta").saveAsTable(
    SILVER_EXTRACTIONS
)

display(silver_extraction_df.limit(5))


In [0]:
# Normalize AI extraction output into Silver obligations, milestones, payments, and risks tables.
obligation_rows = []
milestone_rows = []
payment_rows = []
risk_rows = []

for item in extractions:
    contract_id = item["contract_id"]

    obligation_rows.append(
        (
            contract_id,
            "SUPPLIER_COMMITMENT",
            item.get("commitment"),
            "OPEN"
        )
    )

    for milestone_no, milestone in enumerate(
        item.get("milestones", []),
        start=1
    ):
        milestone_rows.append(
            (
                contract_id,
                milestone_no,
                milestone.get("milestone"),
                milestone.get("target_date"),
                "OPEN"
            )
        )

    for payment in item.get("payments", []):
        payment_rows.append(
            (
                contract_id,
                int(payment.get("payment_no", 0)),
                payment.get("linked_milestone"),
                float(payment.get("amount", 0.0)),
                payment.get("currency"),
                payment.get("due_date"),
                "OPEN"
            )
        )

    risk_rows.append(
        (
            contract_id,
            item.get("risk_type"),
            item.get("risk_severity"),
            item.get("recommended_action"),
            float(item.get("sla_target_pct", 0.0)),
            float(item.get("delay_penalty_pct_per_week", 0.0)),
            int(item.get("renewal_notice_days", 0))
        )
    )

spark.createDataFrame(
    obligation_rows,
    [
        "contract_id",
        "obligation_type",
        "obligation_text",
        "status"
    ]
).write.mode("overwrite").format("delta").saveAsTable(
    SILVER_OBLIGATIONS
)

spark.createDataFrame(
    milestone_rows,
    [
        "contract_id",
        "milestone_no",
        "milestone",
        "target_date",
        "status"
    ]
).write.mode("overwrite").format("delta").saveAsTable(
    SILVER_MILESTONES
)

spark.createDataFrame(
    payment_rows,
    [
        "contract_id",
        "payment_no",
        "linked_milestone",
        "amount",
        "currency",
        "due_date",
        "payment_status"
    ]
).write.mode("overwrite").format("delta").saveAsTable(
    SILVER_PAYMENTS
)

spark.createDataFrame(
    risk_rows,
    [
        "contract_id",
        "risk_type",
        "risk_severity",
        "recommended_action",
        "sla_target_pct",
        "delay_penalty_pct_per_week",
        "renewal_notice_days"
    ]
).write.mode("overwrite").format("delta").saveAsTable(
    SILVER_RISKS
)

display(spark.table(SILVER_RISKS))


In [0]:
# Split Bronze documents into business-clause chunks and persist them as the Silver retrieval corpus.
section_pattern = re.compile(
    r"SECTION\s+(\d+)\s+-\s+([^\n]+)\n(.*?)(?=\nSECTION\s+\d+\s+-|\Z)",
    re.S | re.I
)

clause_rows = []

for contract_id, document_name, contract_text in document_rows:
    matches = section_pattern.findall(
        contract_text
    )

    for section_number, section_title, section_body in matches:
        clause_rows.append(
            (
                f"{contract_id}-S{int(section_number):02d}",
                contract_id,
                int(section_number),
                section_title.strip().upper(),
                section_body.strip(),
                document_name
            )
        )

silver_clause_df = spark.createDataFrame(
    clause_rows,
    [
        "chunk_id",
        "contract_id",
        "section_number",
        "clause_type",
        "clause_text",
        "document_name"
    ]
)

silver_clause_df.write.mode("overwrite").format("delta").option(
    "delta.enableChangeDataFeed",
    "true"
).saveAsTable(
    SILVER_CONTRACT_CLAUSES
)

display(silver_clause_df.limit(20))


## Section 6 — Validate Silver Layer

In [0]:
# Run data-quality and extraction-accuracy checks before Silver data is promoted into Gold.
from pyspark.sql import functions as F

quality_rows = [
    (
        "duplicate_contracts",
        spark.table(BRONZE_CONTRACTS)
        .groupBy("contract_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    ),
    (
        "invalid_payment_amounts",
        spark.table(SILVER_PAYMENTS)
        .filter(F.col("amount") <= 0)
        .count()
    ),
    (
        "missing_milestone_dates",
        spark.table(SILVER_MILESTONES)
        .filter(F.col("target_date").isNull())
        .count()
    ),
    (
        "missing_risk_severity",
        spark.table(SILVER_RISKS)
        .filter(F.col("risk_severity").isNull())
        .count()
    ),
]

quality_df = spark.createDataFrame(
    [
        (
            check_name,
            issue_count,
            issue_count == 0
        )
        for check_name, issue_count in quality_rows
    ],
    [
        "check_name",
        "issue_count",
        "passed"
    ]
)

display(quality_df)

evaluation_rows = []

for item in extractions:
    truth = truth_pdf.loc[
        truth_pdf["contract_id"] == item["contract_id"]
    ].iloc[0]

    first_milestone = (
        item.get("milestones", [{}])[0]
        if item.get("milestones")
        else {}
    )

    first_payment = (
        item.get("payments", [{}])[0]
        if item.get("payments")
        else {}
    )

    evaluation_rows.append(
        (
            item["contract_id"],
            item.get("supplier_name") == truth["supplier_name"],
            first_milestone.get("target_date") == truth["milestone_1_date"],
            abs(
                float(first_payment.get("amount", 0.0))
                - float(truth["payment_1"])
            ) < 0.01,
            item.get("risk_severity") == truth["risk_severity"]
        )
    )

evaluation_df = spark.createDataFrame(
    evaluation_rows,
    [
        "contract_id",
        "supplier_correct",
        "milestone_date_correct",
        "payment_correct",
        "risk_correct"
    ]
)

display(evaluation_df)


## Section 7 — Build Gold Layer

The Gold layer consolidates Bronze master data with Silver contract intelligence.

Target business fields:
- supplier
- contract value
- contract dates
- next milestone
- next payment
- open obligations
- risk
- SLA
- penalty
- renewal exposure
- recommended action


In [0]:
# Build the Gold enterprise contract table from Bronze master data and Silver contract intelligence.
gold_sql = f"""
CREATE OR REPLACE TABLE {GOLD_ENTERPRISE_CONTRACTS}
USING DELTA
AS
WITH next_milestone AS (
    SELECT
        contract_id,
        milestone,
        CAST(target_date AS DATE) AS target_date,
        ROW_NUMBER() OVER (
            PARTITION BY contract_id
            ORDER BY CAST(target_date AS DATE)
        ) AS rn
    FROM {SILVER_MILESTONES}
    WHERE status = 'OPEN'
),
next_payment AS (
    SELECT
        contract_id,
        linked_milestone,
        amount,
        currency,
        CAST(due_date AS DATE) AS due_date,
        ROW_NUMBER() OVER (
            PARTITION BY contract_id
            ORDER BY CAST(due_date AS DATE)
        ) AS rn
    FROM {SILVER_PAYMENTS}
    WHERE payment_status = 'OPEN'
),
obligation_count AS (
    SELECT
        contract_id,
        COUNT(*) AS open_obligations
    FROM {SILVER_OBLIGATIONS}
    WHERE status = 'OPEN'
    GROUP BY contract_id
)
SELECT
    c.contract_id,
    c.supplier_id,
    s.supplier_name,
    s.category,
    s.country,
    s.supplier_tier,

    c.ariba_contract_id,
    a.purchase_order,
    c.source_system,
    c.contract_owner,

    CAST(c.start_date AS DATE) AS start_date,
    CAST(c.end_date AS DATE) AS end_date,
    c.contract_value,
    c.currency,
    c.status,

    nm.milestone AS next_milestone,
    nm.target_date AS next_milestone_date,

    np.linked_milestone AS next_payment_dependency,
    np.amount AS next_payment_amount,
    np.currency AS next_payment_currency,
    np.due_date AS next_payment_due_date,

    COALESCE(oc.open_obligations, 0) AS open_obligations,

    r.risk_type,
    r.risk_severity,
    r.recommended_action,
    r.sla_target_pct,
    r.delay_penalty_pct_per_week,
    r.renewal_notice_days,

    DATEDIFF(
        CAST(c.end_date AS DATE),
        CURRENT_DATE()
    ) AS days_to_expiry,

    CASE
        WHEN DATEDIFF(
            CAST(c.end_date AS DATE),
            CURRENT_DATE()
        ) <= r.renewal_notice_days
        THEN TRUE
        ELSE FALSE
    END AS renewal_review_due

FROM {BRONZE_CONTRACTS} c

JOIN {BRONZE_SUPPLIERS} s
    ON c.supplier_id = s.supplier_id

LEFT JOIN {BRONZE_ARIBA} a
    ON c.contract_id = a.contract_id

LEFT JOIN next_milestone nm
    ON c.contract_id = nm.contract_id
   AND nm.rn = 1

LEFT JOIN next_payment np
    ON c.contract_id = np.contract_id
   AND np.rn = 1

LEFT JOIN obligation_count oc
    ON c.contract_id = oc.contract_id

LEFT JOIN {SILVER_RISKS} r
    ON c.contract_id = r.contract_id
"""

spark.sql(gold_sql)

display(
    spark.table(GOLD_ENTERPRISE_CONTRACTS)
    .orderBy(
        F.col("risk_severity").desc(),
        F.col("next_milestone_date")
    )
)


## Section 8 — Create Gold Materialized View

In [0]:
# Attempt a standalone Gold materialized view and fall back to a standard SQL view when necessary.
MATERIALIZED_VIEW_CREATED = False

try:
    materialized_view_sql = f"""
    CREATE OR REPLACE MATERIALIZED VIEW {GOLD_ENTERPRISE_CONTRACT_VIEW}
    AS
    SELECT *
    FROM {GOLD_ENTERPRISE_CONTRACTS}
    """

    spark.sql(
        materialized_view_sql
    )

    MATERIALIZED_VIEW_CREATED = True

    print(
        "Materialized view created:",
        GOLD_ENTERPRISE_CONTRACT_VIEW
    )

except Exception as e:
    print(
        "Materialized view unavailable; creating a standard Gold view."
    )
    print(
        type(e).__name__,
        str(e)[:500]
    )

    try:
        spark.sql(
            f"DROP VIEW IF EXISTS {GOLD_ENTERPRISE_CONTRACT_VIEW}"
        )
    except Exception:
        pass

    standard_view_sql = f"""
    CREATE VIEW {GOLD_ENTERPRISE_CONTRACT_VIEW}
    AS
    SELECT *
    FROM {GOLD_ENTERPRISE_CONTRACTS}
    """

    spark.sql(
        standard_view_sql
    )

display(
    spark.table(
        GOLD_ENTERPRISE_CONTRACT_VIEW
    ).limit(20)
)


## Section 9 — Test Gold Business Questions

In [0]:
# Run executive-style contract intelligence questions against the Gold enterprise view.
print("HIGH-RISK CONTRACTS")
display(
    spark.table(GOLD_ENTERPRISE_CONTRACT_VIEW)
    .filter(
        F.col("risk_severity") == "HIGH"
    )
    .select(
        "contract_id",
        "supplier_name",
        "next_milestone",
        "next_milestone_date",
        "next_payment_amount",
        "next_payment_currency",
        "risk_type",
        "delay_penalty_pct_per_week",
        "recommended_action"
    )
)

print("RENEWAL REVIEW DUE")
display(
    spark.table(GOLD_ENTERPRISE_CONTRACT_VIEW)
    .filter(
        F.col("renewal_review_due") == True
    )
    .select(
        "contract_id",
        "supplier_name",
        "end_date",
        "days_to_expiry",
        "renewal_notice_days",
        "risk_severity"
    )
    .orderBy(
        "days_to_expiry"
    )
)


## Section 10 — Build Agent-Ready Retrieval Foundation

In [0]:
# Build a local TF-IDF retrieval index over the Silver contract clause corpus for immediate agent testing.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

clause_pdf = (
    spark.table(
        SILVER_CONTRACT_CLAUSES
    )
    .toPandas()
    .reset_index(drop=True)
)

retrieval_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

clause_matrix = retrieval_vectorizer.fit_transform(
    clause_pdf["clause_text"].fillna("")
)

print(
    "Indexed Silver clause chunks:",
    len(clause_pdf)
)


In [0]:
# Define the retrieval function that Lab D can reuse for contract evidence grounding.
def retrieve_contract_evidence(
    query,
    contract_id=None,
    top_k=5
):
    candidate_pdf = clause_pdf.copy()

    if contract_id is not None:
        candidate_pdf = candidate_pdf[
            candidate_pdf["contract_id"] == contract_id
        ].copy()

    if candidate_pdf.empty:
        return pd.DataFrame()

    candidate_indices = (
        candidate_pdf.index.to_list()
    )

    query_vector = retrieval_vectorizer.transform(
        [query]
    )

    scores = cosine_similarity(
        query_vector,
        clause_matrix[candidate_indices]
    ).flatten()

    result_pdf = candidate_pdf.copy()
    result_pdf["score"] = scores

    return (
        result_pdf[
            [
                "score",
                "chunk_id",
                "contract_id",
                "clause_type",
                "clause_text"
            ]
        ]
        .sort_values(
            "score",
            ascending=False
        )
        .head(top_k)
        .reset_index(drop=True)
    )

display(
    retrieve_contract_evidence(
        "What penalty applies when a milestone is delayed?",
        top_k=5
    )
)


## Section 11 — Define the Gold Structured Query Interface

In [0]:
# Define the Gold structured-query interface that Lab D agents can call for enterprise contract facts.
def query_contract_data(
    intent,
    contract_id=None,
    supplier_id=None,
    days=90,
    limit=20
):
    df = spark.table(
        GOLD_ENTERPRISE_CONTRACT_VIEW
    )

    if contract_id is not None:
        df = df.filter(
            F.col("contract_id") == contract_id
        )

    if supplier_id is not None:
        df = df.filter(
            F.col("supplier_id") == supplier_id
        )

    if intent == "contract_detail":
        result_df = df

    elif intent == "high_risk":
        result_df = (
            df.filter(
                F.col("risk_severity") == "HIGH"
            )
            .orderBy(
                "next_milestone_date"
            )
        )

    elif intent == "payments_due":
        result_df = (
            df.filter(
                F.col("next_payment_due_date").isNotNull()
            )
            .filter(
                F.col("next_payment_due_date")
                <= F.date_add(
                    F.current_date(),
                    int(days)
                )
            )
            .orderBy(
                "next_payment_due_date"
            )
        )

    elif intent == "expiring":
        result_df = (
            df.filter(
                (F.col("days_to_expiry") >= 0)
                & (F.col("days_to_expiry") <= int(days))
            )
            .orderBy(
                "days_to_expiry"
            )
        )

    elif intent == "supplier_summary":
        result_df = (
            df.groupBy(
                "supplier_id",
                "supplier_name"
            )
            .agg(
                F.countDistinct(
                    "contract_id"
                ).alias(
                    "contracts"
                ),
                F.sum(
                    "contract_value"
                ).alias(
                    "total_contract_value"
                ),
                F.sum(
                    F.when(
                        F.col("risk_severity") == "HIGH",
                        1
                    ).otherwise(0)
                ).alias(
                    "high_risk_contracts"
                ),
                F.sum(
                    "open_obligations"
                ).alias(
                    "open_obligations"
                )
            )
        )

    else:
        raise ValueError(
            "Supported intents: contract_detail, high_risk, "
            "payments_due, expiring, supplier_summary"
        )

    return result_df.limit(
        int(limit)
    )

display(
    query_contract_data(
        "supplier_summary"
    )
)


## Section 12 — Medallion Lineage Check

The final lineage is:

```text
BRONZE
bronze_suppliers
bronze_contracts
bronze_documents
bronze_ariba
bronze_erp_payments
bronze_sharepoint
        │
        ▼
AI extraction / normalization
        │
        ▼
SILVER
silver_extractions
silver_obligations
silver_milestones
silver_payments
silver_risks
silver_contract_clauses
        │
        ▼
GOLD
gold_enterprise_contracts
        │
        ▼
gold_enterprise_contract_view
        │
        ├── structured agent tool
        └── Silver clause retrieval tool
                 │
                 ▼
              Lab D
```


In [0]:
# Print the complete Bronze, Silver, and Gold object map for use in Lab D.
medallion_summary = {
    "bronze": {
        "suppliers": BRONZE_SUPPLIERS,
        "contracts": BRONZE_CONTRACTS,
        "documents": BRONZE_DOCUMENTS,
        "ariba": BRONZE_ARIBA,
        "erp_payments": BRONZE_ERP_PAYMENTS,
        "sharepoint": BRONZE_SHAREPOINT,
    },
    "silver": {
        "extractions": SILVER_EXTRACTIONS,
        "obligations": SILVER_OBLIGATIONS,
        "milestones": SILVER_MILESTONES,
        "payments": SILVER_PAYMENTS,
        "risks": SILVER_RISKS,
        "contract_clauses": SILVER_CONTRACT_CLAUSES,
    },
    "gold": {
        "enterprise_contracts": GOLD_ENTERPRISE_CONTRACTS,
        "enterprise_contract_view": GOLD_ENTERPRISE_CONTRACT_VIEW,
        "materialized_view_created": MATERIALIZED_VIEW_CREATED,
    },
    "ai": {
        "extraction_mode": EXTRACTION_MODE,
        "llm_endpoint": LLM_ENDPOINT,
    },
    "lab_d_interfaces": {
        "structured_query": "query_contract_data",
        "retrieval": "retrieve_contract_evidence",
    },
}

print(
    json.dumps(
        medallion_summary,
        indent=2
    )
)


## Lab C Completion Checklist

### Bronze
- `bronze_suppliers`
- `bronze_contracts`
- `bronze_documents`
- `bronze_ariba`
- `bronze_erp_payments`
- `bronze_sharepoint`

### AI transformation
- Contract extraction attempted with `ai_query`
- Deterministic fallback included
- Raw extraction JSON retained
- Synthetic ground truth retained for extraction validation

### Silver
- `silver_extractions`
- `silver_obligations`
- `silver_milestones`
- `silver_payments`
- `silver_risks`
- `silver_contract_clauses`

### Silver quality
- Data-quality checks
- Key extraction accuracy checks
- Clause corpus prepared for retrieval

### Gold
- `gold_enterprise_contracts`
- `gold_enterprise_contract_view`
- Materialized-view attempt
- Standard-view fallback

### Agent readiness
- `query_contract_data(...)`
- `retrieve_contract_evidence(...)`

## Next notebook

**Lab D — Agentic Contract Intelligence on Medallion Architecture**

`Gold + Silver Evidence → Supervisor → Contract Agent → Risk Agent → Action Agent → Evals → Quality Gate → Human Approval → Action Queue`
